# Nairobi Business Intelligence & Sales Analytics — Full Analysis Walkthrough

**Author:** Davis Kunyu Wamalwa  
**Purpose:** End-to-end data analysis portfolio project for a simulated Kenyan e-commerce business in Nairobi.

---
## Section 1: Project Introduction

**Business context:** This project simulates a retail/e-commerce business operating in Nairobi. The data includes customers (with Nairobi neighbourhoods), products (6 categories), orders (2024), order line items, payments (including M-Pesa), and customer support tickets.

**Objectives:**
- Load and assess data quality
- Explore revenue, customers, and products
- Run SQL business reports
- Perform statistical tests and segmentation
- Produce a simple sales forecast and business recommendations

---
## Section 2: Database Setup & Data Loading

We connect to the SQLite database and load all six tables into pandas DataFrames. The database path is relative to the project root.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
DB_PATH = BASE / "database" / "nairobi_business.db"

conn = sqlite3.connect(str(DB_PATH))
tables = ["customers", "products", "orders", "order_items", "payments", "customer_support"]
data = {t: pd.read_sql_query(f"SELECT * FROM {t}", conn) for t in tables}
conn.close()

for name, df in data.items():
    print(f"{name}: shape={df.shape}")
    display(df.head())

---
## Section 3: Data Quality Assessment

We check for nulls, duplicates, and basic integrity. This helps decide cleaning steps before analysis.

In [ ]:
for name, df in data.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    dups = df.duplicated().sum()
    print(f"{name}: nulls={nulls.to_dict() if len(nulls) else 'none'}, duplicates={dups}")

---
## Section 4: Exploratory Data Analysis (EDA)

We analyse revenue trends, customer distribution, and product performance. Key charts are generated inline.

In [ ]:
import matplotlib.pyplot as plt

orders = data['orders']
valid_orders = orders[orders['status'].isin(['Delivered', 'Processing', 'Pending'])]
monthly = valid_orders.groupby(pd.to_datetime(valid_orders['order_date']).dt.to_period('M'))['total_amount'].sum()
monthly.plot(kind='bar', figsize=(10,4), title='Monthly Revenue (KSh) 2024')
plt.xticks(rotation=45)
plt.ylabel('Revenue (KSh)')
plt.tight_layout()
plt.show()

In [ ]:
data['customers']['location'].value_counts().plot(kind='bar', figsize=(10,4), title='Customers by Nairobi Location')
plt.xticks(rotation=45)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
oi = data['order_items'].merge(data['products'][['product_id','category']], on='product_id')
oi = oi.merge(data['orders'][['order_id','status']], on='order_id')
rev_cat = oi[oi['status'].isin(['Delivered','Processing','Pending'])].groupby('category')['subtotal'].sum()
rev_cat.plot(kind='pie', autopct='%1.1f%%', figsize=(8,8), title='Revenue by Category')
plt.ylabel('')
plt.tight_layout()
plt.show()

---
## Section 5: SQL Business Reports

We run key SQL queries and display results as DataFrames.

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
monthly_report = pd.read_sql_query("""
SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS orders, 
       ROUND(SUM(total_amount),2) AS revenue, ROUND(AVG(total_amount),2) AS aov
FROM orders WHERE status NOT IN ('Cancelled','Returned') AND order_date >= '2024-01-01'
GROUP BY strftime('%Y-%m', order_date) ORDER BY month
""", conn)
display(monthly_report)
conn.close()

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
top20 = pd.read_sql_query("""
SELECT c.customer_id, c.first_name, c.last_name, c.location,
       COUNT(o.order_id) AS orders, ROUND(SUM(o.total_amount),2) AS revenue
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status NOT IN ('Cancelled','Returned')
GROUP BY c.customer_id ORDER BY revenue DESC LIMIT 20
""", conn)
display(top20)
conn.close()

---
## Section 6: Statistical Analysis

Descriptive statistics and a simple hypothesis test (e.g. H1 vs H2 mean order value).

In [ ]:
from scipy import stats

valid = data['orders'][data['orders']['status'].isin(['Delivered','Processing','Pending'])].copy()
valid['order_date'] = pd.to_datetime(valid['order_date'])
h1 = valid[valid['order_date'].dt.month <= 6]['total_amount']
h2 = valid[valid['order_date'].dt.month >= 7]['total_amount']
t_stat, p_val = stats.ttest_ind(h1, h2)
print(f'Mean H1: {h1.mean():.2f}, Mean H2: {h2.mean():.2f}')
print(f't-test: statistic={t_stat:.4f}, p-value={p_val:.4f}')
print('Interpretation: H1 and H2 differ significantly (reject H0).' if p_val < 0.05 else 'No significant difference.')

---
## Section 7: Customer Segmentation (RFM + K-Means)

We compute Recency, Frequency, Monetary per customer and apply K-Means (k=4) to label segments.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

conn = sqlite3.connect(str(DB_PATH))
rfm = pd.read_sql_query("""
SELECT c.customer_id, 
  JULIANDAY('2024-12-31') - JULIANDAY(MAX(o.order_date)) AS recency,
  COUNT(o.order_id) AS frequency, SUM(o.total_amount) AS monetary
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status NOT IN ('Cancelled','Returned') GROUP BY c.customer_id
""", conn)
conn.close()

X = StandardScaler().fit_transform(rfm[['recency','frequency','monetary']])
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)
rfm['segment_id'] = km.labels_
print(rfm.groupby('segment_id')[['recency','frequency','monetary']].mean().round(2))

---
## Section 8: Sales Forecasting

Simple linear regression on monthly revenue: train on Jan–Sep, predict Oct–Dec and forecast Jan–Mar 2025.

In [ ]:
from sklearn.linear_model import LinearRegression

monthly_rev = valid_orders.groupby(pd.to_datetime(valid_orders['order_date']).dt.to_period('M'))['total_amount'].sum().reset_index()
monthly_rev['month_idx'] = range(len(monthly_rev))
X = monthly_rev[['month_idx']]
y = monthly_rev['total_amount']
model = LinearRegression().fit(X, y)
jan2025 = model.predict([[12]])[0]
print(f'Forecast January 2025 revenue: KSh {jan2025:,.2f}')

---
## Section 9: Business Insights & Recommendations

1. **Segment campaigns:** Target At Risk and Lost segments with win-back offers.
2. **Champions:** Reward high-value recent customers with exclusives.
3. **Delivery & payments:** Reduce support tickets by improving delivery and M-Pesa flows.
4. **Category & location:** Allocate stock and marketing by category and neighbourhood.
5. **Seasonality:** Plan inventory and promotions for Q4 peak.
6. **Monitoring:** Track monthly revenue vs forecast and adjust.
7. **Product mix:** Use basket and category analysis to cross-sell.
8. **CLV:** Use lifetime value for prioritising retention and acquisition.
9. **Support:** Focus on delivery and payment issue types to improve satisfaction.
10. **Testing:** A/B test promotions by segment and channel.

---
## Section 10: Conclusion

This notebook walked through loading data from SQLite, assessing quality, exploring revenue and customer behaviour, running SQL reports, performing statistical tests, segmenting customers with RFM and K-Means, and forecasting sales. Tools used: Python, pandas, SQLite, scipy, scikit-learn, matplotlib. The project demonstrates end-to-end data analyst skills suitable for a portfolio.